# `CTViz`
This notebook is intended to be used for running `Cytetype` and inspecting results.

It should follow `./scripts/train_scvi.py` and `./scripts/cluster`.

In [2]:
# Import libraries
from pathlib import Path
import scanpy as sc
from cytetype import CyteType

## Model Training

In [ ]:
# Import libraries
import scvi
import seaborn as sns

In [ ]:
# Load model
model = scvi.model.SCVI.load(Path.cwd().parent / "models" / "scvi3", adata=adata)

In [ ]:
# Assign model history to variable
hist = model.history

# Set matplotlib aesthetics
def plot_metric(train_key, val_key, title, ylabel):
    plt.figure(figsize=(6, 4))
    plt.plot(hist[train_key], label="train")
    plt.plot(hist[val_key], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Negative ELBO (Evidence lower bound)
if "elbo_train" in hist:
    plot_metric("elbo_train", "elbo_validation", "ELBO", "ELBO")

In [ ]:
# Reconstruction loss
plot_metric(
    "reconstruction_loss_train",
    "reconstruction_loss_validation",
    "Reconstruction loss",
    "Negative log likelihood",
)

In [ ]:
# Local KL divergence
plot_metric(
    "kl_local_train",
    "kl_local_validation",
    "KL divergence (latent z)",
    "KL(q(z|x)||p(z))",
)

## CyteType

In [ ]:
# Load adata
adata = sc.read(
    filename=Path.cwd().parent / "data" / "monocyte_dendritic_cluster.h5ad"
)
adata

In [ ]:
# Plot cell type UMAP
sc.pl.umap(adata, color=["cell_type"], frameon=False)

In [ ]:
# Run CyteType and plot predicted cell type UMAP
group_key = "clusters"
annotator = CyteType(
    adata, group_key=group_key, rank_key="rank_genes_clusters" + group_key, n_top_genes=100
)
adata = annotator.run(study_context="Human PBMC from healthy donors 25-90. Study on the efficacy of influenza vaccines.")
sc.pl.umap(adata, color="cytetype_annotation_clusters", frameon=False)

In [ ]:
# Compare CyteType to annotations in dataset
adata.obs[["cell_type", "cytetype_annotation"]].value_counts()